# Deep Hedging — Phase 4, brique 2 : le benchmark delta sous Heston

On applique le **delta de Black-Scholes** (à volatilité constante, forcément mal spécifiée) pour couvrir un call sur des trajectoires **Heston**. Le message, différent de celui des coûts :

> Sous Heston, même **sans coûts** et en rééquilibrant très finement, l'erreur de couverture ne tend **pas** vers zéro. Elle bute sur un plancher positif : le **risque de vega** que le delta ne peut pas couvrir.

C'est la signature de l'incomplétude. En GBM (marché complet), la même erreur tend vers zéro. Ce plancher est le benchmark faible que le couvreur neuronal va écraser en phase 4 brique 3.

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## Simulateurs, Black-Scholes, couverture (autonome)

In [ ]:
def simulate_heston(S0, v0, mu, kappa, theta, xi, rho, T, n, m):
    dt = T/n; S = np.empty((m, n+1)); v = np.empty((m, n+1)); S[:,0]=S0; v[:,0]=v0
    for k in range(n):
        Z1 = rng.standard_normal(m); Zp = rng.standard_normal(m); Z2 = rho*Z1 + np.sqrt(1-rho**2)*Zp
        vk = np.maximum(v[:,k], 0.0)
        v[:,k+1] = np.maximum(v[:,k] + kappa*(theta-vk)*dt + xi*np.sqrt(vk)*np.sqrt(dt)*Z2, 0.0)
        S[:,k+1] = S[:,k]*np.exp((mu-0.5*vk)*dt + np.sqrt(vk)*np.sqrt(dt)*Z1)
    return S

def simulate_gbm(S0, mu, sig, T, n, m):
    dt = T/n; Z = rng.standard_normal((m, n)); inc = (mu-0.5*sig**2)*dt + sig*np.sqrt(dt)*Z
    return S0*np.exp(np.concatenate([np.zeros((m,1)), np.cumsum(inc, axis=1)], axis=1))

def bs_price(S, K, tau, r, s):
    S = np.asarray(S, float); d1=(np.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)); d2=d1-s*np.sqrt(tau)
    return S*norm.cdf(d1) - K*np.exp(-r*tau)*norm.cdf(d2)

def bs_delta(S, K, tau, r, s):
    S = np.asarray(S, float); return norm.cdf((np.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)))

def cvar(pnl, a=0.95):
    loss = -pnl; return loss[loss >= np.quantile(loss, a)].mean()

def delta_hedge(S, K, T, r, hedge_vol, premium):
    m, n1 = S.shape; n = n1-1; dt = T/n; times = np.linspace(0, T, n1)
    cash = np.full(m, premium); sh = np.zeros(m)
    for k in range(n):
        tau = T - times[k]; dk = bs_delta(S[:,k], K, tau, r, hedge_vol); tr = dk - sh
        cash -= tr*S[:,k]; sh = dk; cash *= np.exp(r*dt)      # sans coûts ici
    return cash + sh*S[:,-1] - np.maximum(S[:,-1]-K, 0.0)

## Prix de l'option et vol de couverture

Le trader encaisse le **vrai prix Heston** de l'option (estimé par Monte-Carlo risque-neutre, drift `r`), en déduit la **vol implicite** Black-Scholes, et couvre avec cette vol constante. C'est exactement ce que faisaient les desks avant les modèles à vol stochastique.

In [ ]:
S0, K, mu, r, T = 100., 100., 0.05, 0.02, 1.0
v0, kappa, theta, xi, rho = 0.04, 2.0, 0.04, 0.3, -0.7   # vol moyenne ~20%, comme le GBM
sig_gbm = 0.20

Sr = simulate_heston(S0, v0, r, kappa, theta, xi, rho, T, 252, 80_000)   # sous r : risque-neutre
C_heston = np.exp(-r*T)*np.mean(np.maximum(Sr[:,-1]-K, 0))
sig_imp = brentq(lambda s: bs_price(S0, K, T, r, s) - C_heston, 1e-3, 2.0)
print(f"prix Heston (MC RN) = {C_heston:.3f}   vol implicite BS = {sig_imp:.4f}")

## Le contraste : Heston bute, GBM tend vers zéro

In [ ]:
ns = [21, 63, 252]
cH, cG = [], []
for n in ns:
    SH = simulate_heston(S0, v0, mu, kappa, theta, xi, rho, T, n, 60_000)
    SG = simulate_gbm(S0, mu, sig_gbm, T, n, 60_000)
    cH.append(cvar(delta_hedge(SH, K, T, r, sig_imp, C_heston)))
    cG.append(cvar(delta_hedge(SG, K, T, r, sig_gbm, bs_price(S0, K, T, r, sig_gbm))))
    print(f"n={n:>4} | CVaR Heston = {cH[-1]:.3f} | CVaR GBM = {cG[-1]:.3f}")

plt.figure(figsize=(7, 4.8))
plt.plot(ns, cH, "o-", color="crimson", label="Heston (vol stochastique)")
plt.plot(ns, cG, "o-", color="navy", label="GBM (vol constante)")
plt.xscale("log"); plt.xlabel("n (rééquilibrages)"); plt.ylabel("CVaR 95% (sans coûts)")
plt.title("Delta-hedging : GBM -> 0, Heston bute sur le risque de vega")
plt.legend(); plt.grid(alpha=0.3); plt.ylim(0, 6.5); plt.tight_layout(); plt.show()

## Lecture et suite

La CVaR Heston (~4 à n=252) ne descendra pas vers zéro : environ 1 point vient de la discrétisation (comme le GBM) et le reste, ~3.9, est le **plancher de vega irréductible** avec le seul sous-jacent. Pour aller sous ce plancher, deux voies : donner au couvreur l'information de variance `v_t` pour qu'il choisisse une meilleure position (le *minimum-variance delta*, qui corrige le delta BS d'un terme en `rho`, `xi`), ou ajouter une seconde option comme instrument. La brique 3 fait la première : on entraîne le réseau avec l'état augmenté `(S, tau, position, v)`, et on s'attend à ce qu'il batte nettement 4.